In [5]:
import pandas as pd
df = pd.read_csv('../data/clean_data.csv')
df.head(5)

,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Time_Orderd,Time_Order_picked,Order_Date,Weatherconditions,Road_traffic_density,Type_of_order,Type_of_vehicle,multiple_deliveries,City,Time_taken(min),distance,order_making(min),delivery_time(min),time_category
0,22.745049,75.892471,22.765049,75.912471,11:30:00,11:45:00,19-03-2022,Sunny,High,Snack,motorcycle,0,Urban,24,3.020737,15,9,Morning
1,12.913041,77.683237,13.043041,77.813237,19:45:00,19:50:00,25-03-2022,Stormy,Jam,Snack,scooter,1,Metropolitian,33,20.143737,5,28,Evening
2,12.914264,77.678400,12.924264,77.688400,08:30:00,08:45:00,19-03-2022,Sandstorms,Low,Drinks,motorcycle,1,Urban,26,1.549693,15,11,Morning
3,11.003669,76.976494,11.053669,77.026494,18:00:00,18:10:00,05-04-2022,Sunny,Medium,Buffet,motorcycle,1,Metropolitian,21,7.774497,10,11,Evening
4,12.972793,80.249982,13.012793,80.289982,13:30:00,13:45:00,26-03-2022,Cloudy,High,Snack,scooter,1,Metropolitian,30,6.197898,15,15,Afternoon


In [6]:
import pandas as pd

target_col = 'Time_taken(min)'

# 1. Target
y = df[target_col]

# 2. Features (Drop target AND leakage columns)
leakage_cols = [target_col, 'order_making(min)', 'delivery_time(min)']
X = df.drop(columns=[col for col in leakage_cols if col in df.columns])

# 3. Extract time features
X['order_hour'] = pd.to_datetime(X['Time_Orderd'], format='%H:%M:%S', errors='coerce').dt.hour
X['order_minute'] = pd.to_datetime(X['Time_Orderd'], format='%H:%M:%S', errors='coerce').dt.minute
X['picked_hour'] = pd.to_datetime(X['Time_Order_picked'], format='%H:%M:%S', errors='coerce').dt.hour
X['picked_minute'] = pd.to_datetime(X['Time_Order_picked'], format='%H:%M:%S', errors='coerce').dt.minute

X = X.drop(columns=['Time_Orderd', 'Time_Order_picked'])

# 4. Extract date features
X['Order_Date'] = pd.to_datetime(X['Order_Date'], dayfirst=True, errors='coerce')
X['order_day'] = X['Order_Date'].dt.day
X['order_month'] = X['Order_Date'].dt.month
X['order_year'] = X['Order_Date'].dt.year

X = X.drop(columns=['Order_Date'])

# 5. One-hot encode categorical columns
X_encoded = pd.get_dummies(
    X,
    columns=[
        'Weatherconditions',
        'Road_traffic_density',
        'Type_of_order',
        'Type_of_vehicle',
        'City',
        'time_category'
    ],
    drop_first=True,
    dtype=int
)

X_encoded.head()

,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,multiple_deliveries,distance,order_hour,order_minute,picked_hour,picked_minute,...,Type_of_order_Drinks,Type_of_order_Meal,Type_of_order_Snack,Type_of_vehicle_motorcycle,Type_of_vehicle_scooter,City_Semi-Urban,City_Urban,time_category_Evening,time_category_Morning,time_category_Night
0,22.745049,75.892471,22.765049,75.912471,0,3.020737,11,30,11,45,...,0,0,1,1,0,0,1,0,1,0
1,12.913041,77.683237,13.043041,77.813237,1,20.143737,19,45,19,50,...,0,0,1,0,1,0,0,1,0,0
2,12.914264,77.678400,12.924264,77.688400,1,1.549693,8,30,8,45,...,1,0,0,1,0,0,1,0,1,0
3,11.003669,76.976494,11.053669,77.026494,1,7.774497,18,0,18,10,...,0,0,0,1,0,0,0,1,0,0
4,12.972793,80.249982,13.012793,80.289982,1,6.197898,13,30,13,45,...,0,0,1,0,1,0,0,0,0,0


In [7]:
# @title Default title text
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# Verify the shapes
print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)

Training features shape: (30775, 31)
Testing features shape: (7694, 31)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# Define the hyperparameters to test
param_grid = {
    'n_estimators': [100, 200],
    'max_features': [0.7, 1.0],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_depth': [10, 20]
}

# Initialize Random Forest
model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

# Grid Search
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

# Train and search
grid_search.fit(X_train, y_train)

# Best model
model = grid_search.best_estimator_

print("Best parameters:")
print(grid_search.best_params_)

/home/houssam/Documents/rihab/DeliveryETA/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/houssam/Documents/rihab/DeliveryETA/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/houssam/Documents/rihab/DeliveryETA/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn

Best parameters:
{'max_depth': 10, 'max_features': 0.7, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}


In [9]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# TRAIN
train_mae = mean_absolute_error(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_r2 = r2_score(y_train, y_train_pred)

# TEST
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_test_pred)

print("TRAINING SET")
print(f"MAE:  {train_mae:.2f} minutes")
print(f"MSE:  {train_mse:.2f}")
print(f"RMSE: {train_rmse:.2f} minutes")
print(f"R²:   {train_r2:.4f}")

print("\nTEST SET")
print(f"MAE:  {test_mae:.2f} minutes")
print(f"MSE:  {test_mse:.2f}")
print(f"RMSE: {test_rmse:.2f} minutes")
print(f"R²:   {test_r2:.4f}")

# # 1. Predict on the test set
# y_pred = model.predict(X_test)

# # 2. Calculate evaluation metrics
# mae = mean_absolute_error(y_test, y_pred)
# mse = mean_squared_error(y_test, y_pred)
# rmse = np.sqrt(mse)
# r2 = r2_score(y_test, y_pred)

# # 3. Display the results
# print(f"MAE  (Mean Absolute Error):      {mae:.2f} minutes")
# print(f"MSE  (Mean Squared Error):       {mse:.2f}")
# print(f"RMSE (Root Mean Squared Error):  {rmse:.2f} minutes")
# print(f"R² Score:                        {r2:.4f}")

TRAINING SET
MAE:  4.54 minutes
MSE:  33.95
RMSE: 5.83 minutes
R²:   0.6132

TEST SET
MAE:  4.86 minutes
MSE:  37.68
RMSE: 6.14 minutes
R²:   0.5701
